# 02 隱私優先：每一列都是鄰居 — 筆記本

在 Maya 把任何東西分享給 PESU 的 Doc Mon 之前，她必須把 Jun 的原始匯出檔，變成一份能保護檔案中每個人的檔案。在這本筆記本裡，你會：

1. 把欄位分成**直接識別資訊**、**間接識別資訊**和**敏感的健康資訊**；
2. 刪除直接識別資訊，並抓出 Tala 在備註裡**漏掉的一個姓名**；
3. 看看為什麼單純的**姓名雜湊並不匿名**，並做出**加鹽**的假名 ID；
4. 將年齡和日期**泛化**；
5. 在一份你打算公開的表格裡，**隱匿小數字**；
6. 計算有多少人是唯一的，藉此衡量**再識別風險**。

```{warning}
**`santa_luz_linelist_raw.csv` 裡的姓名、電話和住址都是假的。**
它們是電腦編出來的，故意做得*看起來*像真的，好讓你練習。Santa Luz 市與其中的每個人都是虛構的。真正的 line list 絕不會離開你單位受保護的電腦，也絕不會進入你要分享的筆記本或 AI 聊天。
```

這是訓練用的良好做法，不是法律意見。實際工作時，請依照你單位的資料保護長（Data Protection Officer）以及適用於你的 DOH 規定。

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 1. 打開 Jun 的原始匯出檔

檔案的第一行是一則說明資料為合成資料的註記，所以我們用 `skiprows=1` 略過它。`dtype=str` 讓每個儲存格都維持打字時的原樣文字，`keep_default_na=False` 讓空白的儲存格維持為空字串 `""`，而不是變成「缺漏」。

In [ ]:
import pandas as pd

raw = pd.read_csv(
    "data/synthetic/ph/santa_luz_linelist_raw.csv",
    skiprows=1, dtype=str, keep_default_na=False,
)
print(raw.shape)
raw.head(3)      # 假的姓名、電話與住址

## 2. 哪些欄位會識別出一個人？

- **直接識別資訊**單獨就能指向一個人：姓名、電話號碼、街道住址、PhilHealth 或其他證號。
- **間接識別資訊**（*準識別碼*）單獨不會點出任何人，但幾個合在一起就可以：barangay + 年齡 + 性別 + 發病日，可能只對得上一個人。
- **敏感的健康資訊**是這一列「關於什麼」：疾病、這個人是否住院、是否死亡。
- **自由文字**（`Remarks`）可以包含以上任何一種。在你讀過每一筆之前，都當成直接識別資訊處理。

把這些角色寫成一個字典有額外的好處：我們可以**檢查**每個欄位都被分到了角色，這樣就不會有任何欄位漏網而未被分類。

In [ ]:
roles = {
    "Patient Name": "直接識別資訊",
    "Contact No.": "直接識別資訊",
    "Address": "直接識別資訊",
    "Remarks": "自由文字（當成直接識別資訊處理）",
    "Reporting Unit": "間接識別資訊",
    "Barangay": "間接識別資訊",
    "Age": "間接識別資訊",
    "Sex": "間接識別資訊",
    "Date Reported": "間接識別資訊",
    "Date of Onset": "間接識別資訊",
    "Date Consulted": "間接識別資訊",
    "Date Lab Confirmed": "間接識別資訊",
    "Disease": "敏感的健康資訊",
    "Classification": "敏感的健康資訊",
    "Admitted": "敏感的健康資訊",
    "Outcome": "敏感的健康資訊",
    "Dengue Severity": "敏感的健康資訊",
    "Waded in Floodwater": "敏感的健康資訊",
    "Did Cleanup Work": "敏感的健康資訊",
}

# 檢查：每個欄位都有角色，而且沒有角色被分給不存在的欄位
assert set(roles) == set(raw.columns), set(raw.columns) ^ set(roles)
print(pd.Series(roles).value_counts())

## 3. 問問 Tala：給結構，不給資料列

Maya **沒有**貼上任何資料列。她只描述欄位：

> *我有一個 pandas DataFrame `raw`，有 19 個文字欄位：Date Reported, Reporting Unit,
> Patient Name, Sex, Age, Contact No., Address, Barangay, Disease, Date of Onset,
> Date Consulted, Classification, Date Lab Confirmed, Admitted, Outcome,
> Waded in Floodwater, Did Cleanup Work, Dengue Severity, Remarks。
> 我不能分享任何資料列：它們是機密的病患資料。
> 請寫 pandas 程式碼，移除直接識別資訊，這樣我才能把表格分享給我們的省級流行病學家。*

Tala 的回答是：

In [ ]:
# Tala 的程式碼
direct_identifiers = ["Patient Name", "Contact No.", "Address"]
shared = raw.drop(columns=direct_identifiers)

print(shared.shape)
print(list(shared.columns))

## 4. 檢查一下：檔案裡還有姓名嗎？

Tala 的程式碼執行時沒有錯誤，那三個欄位也消失了。但「沒有錯誤」不等於「沒有姓名」。一個簡單的測試：把剛剛刪掉的那一欄裡每個**名字（first name）**取出來，在**所有剩下的儲存格**裡搜尋它。搜尋模式 `\b(Ana|Dennis|...)\b` 的意思是「這些名字之一，且是完整的單字」。

In [ ]:
first_names = sorted(set(raw["Patient Name"].str.split().str[0]))
pattern = r"\b(?:" + "|".join(first_names) + r")\b"

hits = shared.apply(lambda column: column.str.contains(pattern)).sum()
print("依欄位列出含有名字的儲存格數：")
print(hits[hits > 0])

shared.loc[shared["Remarks"].str.contains(pattern), "Remarks"].head(5)

### Tala 搞錯了

仍有四十筆備註點名了某個人，例如 *"anak/kapatid ni Dennis (kapitbahay)"*：Dennis 的孩子或手足，鄰居。這則備註點名了**別人**，再加上 barangay，就可以直接指向一個家戶。其他備註也有風險：*"estudyante, Grade 5"*（學生，五年級）把一個孩子縮小到某一個班級。

Tala 移除了名稱*叫做*姓名、電話和住址的欄位。它看不到 `Remarks` 裡面，因為 Maya（做得很對）從沒給它看過任何資料列。**模型只知道你告訴它的事；檢查輸出是你的工作。**

修正方法：也刪掉 `Remarks`（裡面任何有用的資訊，都應該先在單位內編碼成是／否欄位），並刪掉 `Reporting Unit`，因為有些通報單位只送過一兩份報告，單位名稱就可能指向某個人。然後再做一次同樣的檢查：結果必須是**零**。

In [ ]:
remove = ["Patient Name", "Contact No.", "Address", "Remarks", "Reporting Unit"]
shared = raw.drop(columns=remove)

hits = shared.apply(lambda column: column.str.contains(pattern)).sum()
assert hits.sum() == 0, "共用的表格裡還有姓名！"
print(f"檢查了 {shared.shape[0]} 列 x {shared.shape[1]} 欄：已經沒有任何名字。")
print("各單位的通報數（單位名稱為什麼有風險）：")
print(raw["Reporting Unit"].value_counts().tail(3))

## 5. 假名 ID：為什麼單純的雜湊不夠

我們仍然需要一個 ID，這樣 Doc Mon 才能說*「再查一下第 17 號個案」*，Maya 也才找得到它。**雜湊（hash）**把文字變成一串固定的亂碼；同樣的文字永遠得到同樣的亂碼。很多人以為把姓名雜湊就匿名了。其實不是：任何手上有**常見姓名清單**（選民名冊、班級名單、社群媒體）的人，都能把清單上的每個姓名雜湊後找比對。下面的儲存格用一份只有五個名字的虛構清單，扮演這個「攻擊者」。

In [ ]:
import hashlib


def plain_hash(name: str) -> str:
    return hashlib.sha256(name.encode()).hexdigest()[:12]


leaked_id = plain_hash("Juan P. Santos")          # 未加鹽的檔案裡會出現的內容
print("共用檔案裡的 ID：", leaked_id)

guess_list = ["Maria L. Reyes", "Jose A. Cruz", "Juan P. Santos", "Ana B. Garcia", "Mark D. Lim"]
for guess in guess_list:
    if plain_hash(guess) == leaked_id:
        print("攻擊者找到了這個人：", guess)

**加上一個秘密的鹽。** 我們用 `hmac` 把一串長長的隨機秘密（*鹽*）混進雜湊裡。沒有鹽，沒有人能重做雜湊，所以猜測清單就沒用了。單位把鹽存在**單位內**一個上鎖的檔案裡，絕不放進筆記本、絕不放進 AI 聊天、絕不跟資料一起寄出。每週都使用同一個鹽，表示同一個人在不同次匯出中會保有同一個 ID。

雜湊之前，我們先把姓名**正規化**（大寫、去掉句點、空白只留一個），讓 `Juan P. Santos` 和 `JUAN P SANTOS` 得到同一個 ID。最後的檢查正是測試這一點，以及不同的鹽會給出不同的 ID。

`secrets.token_hex(16)` 每次執行都會產生新的隨機鹽，所以**你的 ID 會和其他人的不同**，這正是重點。

In [ ]:
import hmac
import secrets

SALT = secrets.token_hex(16)     # 僅為示範：實際工作時，只建立一次並鎖起來


def pseudo_id(name: str, salt: str) -> str:
    key = " ".join(name.upper().replace(".", " ").split())   # 將姓名正規化
    digest = hmac.new(salt.encode(), key.encode(), hashlib.sha256).hexdigest()
    return "P-" + digest[:10]


print(pseudo_id("Juan P. Santos", SALT))

# 檢查
assert pseudo_id("Juan P. Santos", SALT) == pseudo_id("JUAN P SANTOS ", SALT)
assert pseudo_id("Juan P. Santos", SALT) != pseudo_id("Juan P. Santos", "another salt")
assert all(pseudo_id(g, SALT) != leaked_id for g in guess_list)

shared.insert(0, "pseudo_id", raw["Patient Name"].map(lambda n: pseudo_id(n, SALT)))
shared.head(3)

## 6. 泛化：年齡組與週

精確的年齡和精確的日期，是很強的間接識別資訊。對於要離開辦公室的表格，我們把它們**粗略化**：年齡 → 年齡組，日期 → 週。

原始的年齡和日期仍然很雜亂（`8 mos`、`24-Jul-26`、`46227`）；第 03 章會把它們清理乾淨。所以從這裡開始，我們使用 **`santa_luz_linelist.csv`**，也就是第 03 章產出的、已清理且已去識別化的檔案（流水號 ID `SLC-0001`……，沒有姓名、電話、住址、通報單位或備註）。

`pd.cut` 把年齡分到各個區間。`right=False` 表示每個區間包含它的左端點，所以 5 歲會進入 `5-14`，而不是 `0-4`。**檢查**：對每一個有年齡的列，我們分的組別必須和檔案自己的 `age_group` 欄位一致。

In [ ]:
df = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

labels = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]
df["my_age_group"] = pd.cut(df["age"], bins=[0, 5, 15, 25, 45, 65, 200],
                            right=False, labels=labels)

has_age = df["age"].notna()
assert (df.loc[has_age, "my_age_group"].astype(str) == df.loc[has_age, "age_group"]).all()
print(f"所有 {has_age.sum()} 個有年齡的病例，年齡組都一致"
      f"（{(~has_age).sum()} 筆年齡缺漏）。")

# 日期 -> 發病的 ISO 週（週一到週日）
df["onset_week"] = pd.to_datetime(df["date_onset"]).dt.isocalendar().week
print(df[["date_onset", "onset_week"]].head(3))

## 7. 公開之前的小數字隱匿

一份公開的表格裡只要有一個 **1**，它就可能是一個人。*「Buhangin 有一例登革熱死亡」*是整個 barangay 都能對上名字的消息。常見的規則是把 **1 到 4** 的數字隱藏起來，改顯示 `<5`（零維持為零：它不會洩漏任何人）。

我們在最敏感的表格上試試：依 barangay 與疾病的死亡數。還有一個陷阱：如果某一欄**只有一個**被隱藏的儲存格，任何知道該欄總數的人（而全市總數通常是公開的）都可以把看得到的儲存格減掉，把它還原出來。所以這個儲存格也會**檢查**每一欄：只有一個被隱藏的儲存格，表示那一欄不能依 barangay 公開。對這樣的欄位，我們只公布全市總數，不列出 barangay。就連全市總數是 1，也是敏感的：要不要公布，是 Doc Mon 和單位資料保護長要做的決定，不是程式碼能決定的。

In [ ]:
def suppress(table: pd.DataFrame, threshold: int = 5) -> pd.DataFrame:
    if threshold < 1:
        raise ValueError("threshold 至少要是 1")
    small = (table > 0) & (table < threshold)
    return table.astype(object).mask(small, f"<{threshold}")


deaths = df[df["outcome"] == "died"]
death_table = pd.crosstab(deaths["barangay"], deaths["disease"])
print(f"死亡數：{len(deaths)}，分布在 {len(death_table)} 個 barangay")

safe_table = suppress(death_table)
print(safe_table)

# 檢查：剛好只有「一個」被隱藏儲存格的欄位，可以從它的總數還原
hidden = (safe_table == "<5").sum()
recoverable = list(hidden[hidden == 1].index)
print("只有一個被隱藏儲存格的欄位：", recoverable)

publishable = safe_table.drop(columns=recoverable)
assert ((publishable == "<5").sum() != 1).all()
print(publishable)
for column in recoverable:
    print(f"{column}：只公布全市總數（{death_table[column].sum()}），不列出 barangay。")

## 8. 再識別檢查：有多少人是唯一的？

如果一列在它的間接識別資訊上是**唯一**的，它就有風險，也就是檔案裡沒有任何其他人和它有同樣的 barangay、年齡、性別、疾病、發病日……。**k-匿名性（k-anonymity）**的想法是：每一種組合都該至少被 *k* 個人共有（比如 *k* = 5），這樣任何一列都可能是至少五個人當中的任何一個。

這個函式為每一列計算有多少列與它有相同的組合（*k*），並回報有多少列是唯一的（*k* = 1）。我們嘗試四種細節程度，從精確到粗略。在所選欄位中有缺漏值的列，不計入各次計數。

In [ ]:
import matplotlib.pyplot as plt
# BEGIN-CJK (中文字型設定：避免標籤顯示為方框)
import pathlib
import matplotlib.font_manager as fm
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass
_discovered = []
for _entry in fm.fontManager.ttflist:
    if any(_kw in _entry.name.lower() for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = ["Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP", "Noto Sans TC",
              "Microsoft JhengHei", "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
              "Heiti TC", "DejaVu Sans"]
_cjk_fonts = _discovered + [n for n in _preferred if n not in _discovered]
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts
# END-CJK


def uniqueness(data: pd.DataFrame, quasi: list[str]) -> dict:
    rows = data.dropna(subset=quasi)
    k = rows.groupby(quasi)[quasi[0]].transform("size")   # 每一列所在組別的大小
    return {"列數": len(rows), "唯一（k=1）": int((k == 1).sum()), "k<5": int((k < 5).sum())}


levels = {
    "barangay + 年齡 + 性別 + 疾病 + 發病日": ["barangay", "age", "sex", "disease", "date_onset"],
    "barangay + 年齡 + 性別 + 疾病": ["barangay", "age", "sex", "disease"],
    "barangay + 年齡組 + 性別 + 疾病": ["barangay", "age_group", "sex", "disease"],
    "地帶 + 年齡組 + 性別 + 疾病": ["band", "age_group", "sex", "disease"],
}
risk = pd.DataFrame({name: uniqueness(df, cols) for name, cols in levels.items()}).T
risk["唯一的百分比"] = (risk["唯一（k=1）"] / risk["列數"] * 100).round(1)
print(risk)

ax = risk["唯一的百分比"].iloc[::-1].plot.barh(color="#CE1126", figsize=(8, 3.2))
ax.set_xlabel("在這些欄位上唯一的病例百分比（k = 1）")
ax.set_title("欄位越粗略，唯一的人越少（Santa Luz，虛構）")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## 換你試試

1. **欄位越多，風險越高。** 把 `"case_class"` 加進下面的 `my_columns`，然後執行儲存格。唯一的列數會變多還是變少？*提示：* 每多一個欄位，只會把組別切得更小。
2. **更嚴格的門檻。** 有些單位會隱匿小於 10 的數字。執行 `suppress(death_table, threshold=10)`（已經在儲存格裡），讀讀結果：表格還剩多少是有用的？
3. **寫一個安全的提示詞。** 在文字儲存格裡，寫一個提示詞，請你的 AI 助理寫出依 `barangay` 和 `age_group` 計算登革熱病例數、並隱匿小於 5 的儲存格的程式碼。只使用欄位名稱，以及兩三列明顯是編造的資料（例如 barangay 用 `"Barangay X"`）。然後把它的程式碼和上面的 `suppress()` 比較。*提示：* 檢查它的程式碼有沒有放過零。

In [ ]:
# --- 換你試試 1 ---
my_columns = ["band", "age_group", "sex", "disease"]      # <- 加上 "case_class"
print(uniqueness(df, my_columns))

# --- 換你試試 2 ---
suppress(death_table, threshold=10)

## 帶走重點（Baon）

- **每一欄的裡面都要看。** 只刪除名稱*叫做*姓名、電話和住址的欄位是不夠的：自由文字的備註點名了另外 40 個人。要在結果裡搜尋你刪掉的東西。
- **用秘密做假名化。** 單純的姓名雜湊，可以用猜測清單反推出來；加鹽的 ID（鹽鎖在單位內），或搭配上鎖對照表的流水號 ID，則不行。
- **粗略化、隱匿，然後衡量。** 年齡組、週，以及 `<5` 的儲存格，會大幅降低唯一的人所占的比例，而唯一性檢查會告訴你做得夠不夠。

下一站：[第 03 章，*從雜亂的造冊到乾淨的資料*](../03_clean_linelist.md)。